# Solutions · Chapter 05-10 · Decision trees and random forests

Worked answers to every exercise in `notebooks/05_regression/05-10_trees.ipynb`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")


def true_curve(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5

# SYNTHETIC: the chapter's 400 curve points, noise sd 1.5
curve_rng = np.random.default_rng(7)
curve_x = curve_rng.uniform(-4, 4, 400)
curve_y = true_curve(curve_x) + curve_rng.normal(0, NOISE_SD, 400)
fit_x, held_x, fit_y, held_y = train_test_split(curve_x, curve_y, test_size=0.5,
                                                random_state=0)


def curve_rmse(model, x, y):
    return float(np.sqrt(((y - model.predict(x.reshape(-1, 1))) ** 2).mean()))


print("%d rows to fit, %d held out, noise floor %.2f"
      % (len(fit_x), len(held_x), NOISE_SD))

## Quick understanding

### E1 · Leaves and splits

**A leaf predicts the mean of the training targets that fall in it.** That is the entire model output -
one number per leaf.

**A split is chosen by trying every candidate threshold and keeping the one that minimises the total
within-group squared error**, `SSE(left) + SSE(right)`. Since a group's SSE about its own mean is `n`
times its variance, this is equivalent to maximising the variance *removed* by the split - which is why
the same criterion is sometimes called variance reduction.

The search is exhaustive rather than clever: with `n` distinct values there are `n - 1` candidate
thresholds per feature, and the algorithm evaluates all of them.

### E2 · Two things a tree gets for free

**Non-linearity.** 05-05 had to add `stops_squared` by hand to remove a ∪ from the residuals. A tree
approximates any shape by stacking steps, with no terms supplied.

**Interactions.** 05-07 had to add `promo x weekend`. A tree splits on `weekend` and then on `promo`
*inside* each branch, which is an interaction by construction - the chapter's depth-5 tree reached RMSE
27.94 against the main-effects linear model's 40.65, having been told nothing.

**A third, worth naming: no scaling.** The algorithm only ever compares a feature against a threshold, so
any monotone transform of that feature gives the identical tree. This is the requirement that 05-06 and
05-09 called non-negotiable, and here it simply does not exist.

### E3 · Why averaging kills variance and not bias

**Variance is the spread of the fits about their own average.** Averaging `k` of them is computing that
average, and the spread of an average is smaller than the spread of one - so the term shrinks.

**Bias is the distance of that average from the truth.** Averaging more fits estimates the same average
more precisely; it does not move it.

The chapter measured both: bias² went 0.0189 to 0.0097 (unchanged, within noise) while variance went
2.2780 to 1.0624. **This is why bagging is applied to trees rather than to linear models** - a tree is the
low-bias, high-variance model the trick was designed for, and a linear model has the opposite profile.

## Hand calculation

### E4 · Every split on four rows

`x = [1, 2, 5, 6]`, `y = [10, 12, 20, 22]`. The overall mean is 16, so the total SSE is
`36 + 16 + 16 + 36 = 104`.

| threshold | left | right | left mean | right mean | SSE left | SSE right | total | reduction |
|---|---|---|---|---|---|---|---|---|
| 1.5 | {10} | {12, 20, 22} | 10 | 18 | 0 | 56 | 56 | 48 |
| **3.5** | {10, 12} | {20, 22} | 11 | 21 | **2** | **2** | **4** | **100** |
| 5.5 | {10, 12, 20} | {22} | 14 | 22 | 56 | 0 | 56 | 48 |

**The winner is 3.5, leaving an SSE of 4 out of 104 - it removes 96% of the squared error.**

Working for the winning row: left mean `(10 + 12)/2 = 11`, so SSE is `1 + 1 = 2`; right mean
`(20 + 22)/2 = 21`, so SSE is `1 + 1 = 2`. For the 1.5 row, the right group's mean is
`(12 + 20 + 22)/3 = 18` and its SSE is `36 + 4 + 16 = 56`.

**The two outer splits tie at 56**, which is worth noticing: both leave one group of three containing the
gap between 12 and 20, and that gap is where all the error lives.

### E5 · Predictions at x = 3 and x = 100

The depth-1 tree splits at 3.5 and predicts the group means: **11** on the left, **21** on the right.

- **At x = 3:** 3 < 3.5, so the answer is **11**.
- **At x = 100:** 100 >= 3.5, so the answer is **21**.

**The second is the chapter's failure lab in miniature.** x = 100 is sixteen times beyond any training
value and the tree returns the mean of `{20, 22}` - the same number it would return at x = 6, x = 1,000
or x = 10^9. There is no leaf for territory it has never seen, so the outermost one absorbs everything.

### E6 · Splitting a leaf

Leaf `{4, 6, 6, 8, 11}`, mean `35 / 5 = ` **7**.

`SSE = 9 + 1 + 1 + 1 + 16 = ` **28**.

Split into `{4, 6, 6}` and `{8, 11}`:

- Left mean `16/3 = 5.333`; SSE `= 1.778 + 0.444 + 0.444 = ` **2.667**
- Right mean `9.5`; SSE `= 2.25 + 2.25 = ` **4.5**
- Total **7.167**, a reduction of **20.833** - 74% of the leaf's error, from one question.

**And the mechanical point:** the split improves the fit *on these five rows* by construction. It always
does; the question is only whether it improves the fit on rows the tree has not seen, which is what
`min_samples_leaf` and `max_depth` are guessing about.

### E7 · 500 rows, no limits

**Up to 500 leaves - one per row - and a training RMSE of exactly 0.0000.**

The tree keeps splitting until every leaf is pure, and a leaf holding one row is pure by definition, so it
predicts that row's target exactly. The chapter's version did this at 200 rows and 200 leaves.

Fewer than 500 leaves only if some rows share identical feature values but different targets, in which
case they cannot be separated and the training RMSE is above zero.

**The practical reading: `max_depth=None` is not a neutral default.** It is a decision to memorise, and
scikit-learn makes it silently.

## Coding

### E8 · Best split from scratch

In [ ]:
def best_split(x, y):
    order = np.argsort(x)
    x, y = np.asarray(x)[order], np.asarray(y)[order]
    total = float(((y - y.mean()) ** 2).sum())
    best_threshold, best_sse = None, np.inf
    for index in range(len(x) - 1):
        if x[index] == x[index + 1]:
            continue                       # no split between equal values
        threshold = (x[index] + x[index + 1]) / 2
        left, right = y[x < threshold], y[x >= threshold]
        sse = float(((left - left.mean()) ** 2).sum()
                    + ((right - right.mean()) ** 2).sum())
        if sse < best_sse:
            best_threshold, best_sse = threshold, sse
    return best_threshold, total - best_sse


checks = []
for seed, size in [(0, 40), (1, 200), (2, 15)]:
    rng = np.random.default_rng(seed)
    x = rng.uniform(-3, 3, size)
    y = np.where(x > 0.4, 5.0, -1.0) + rng.normal(0, 1.0, size)
    mine, reduction = best_split(x, y)
    theirs = DecisionTreeRegressor(max_depth=1).fit(x.reshape(-1, 1), y).tree_.threshold[0]
    checks.append({"rows": size, "my threshold": mine, "sklearn": theirs,
                   "difference": abs(mine - theirs), "SSE reduction": reduction})
print(pd.DataFrame(checks).to_string(index=False, float_format=lambda v: "%.6f" % v))

**Identical on all three, to the last decimal place.**

Two details that matter in the implementation and are easy to miss.

**Skipping equal adjacent values.** A threshold between two identical `x` values would put them in
different groups, which is impossible - the comparison `x < threshold` cannot separate them. Without the
guard the loop silently evaluates a split that does not exist.

**The midpoint convention.** The threshold is placed halfway between the two values rather than at either
one. Any value in the open interval gives the same partition of the training rows, and the midpoint is
the choice that is most robust for rows the tree has not seen - which is exactly what sklearn does, and
why the two agree to six decimals rather than merely to the same partition.

### E9 · `min_samples_leaf` against `max_depth`

In [ ]:
leaf_rows = []
for minimum in [1, 2, 3, 5, 8, 12, 20, 35, 50]:
    tree = DecisionTreeRegressor(min_samples_leaf=minimum, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    leaf_rows.append({"min_samples_leaf": minimum, "leaves": tree.get_n_leaves(),
                      "train RMSE": curve_rmse(tree, fit_x, fit_y),
                      "held-out RMSE": curve_rmse(tree, held_x, held_y)})
leaf_table = pd.DataFrame(leaf_rows)
print(leaf_table.to_string(index=False, float_format=lambda v: "%.4f" % v))

winner = leaf_table.loc[leaf_table["held-out RMSE"].idxmin()]
print("\nbest min_samples_leaf %d: held-out RMSE %.4f with %d leaves"
      % (winner["min_samples_leaf"], winner["held-out RMSE"], winner["leaves"]))
print("the chapter's best max_depth was 3: held-out RMSE 1.5867 with 8 leaves")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(leaf_table["min_samples_leaf"], leaf_table["train RMSE"], "o-", color="#0072B2",
        linewidth=2.2, markersize=8, label="on the fitting rows")
ax.plot(leaf_table["min_samples_leaf"], leaf_table["held-out RMSE"], "s-",
        color="#D55E00", linewidth=2.2, markersize=8, label="on held-out rows")
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.6,
           label="noise floor, %.2f" % NOISE_SD)
ax.axhline(1.5867, color="#009E73", linestyle=":", linewidth=2,
           label="best max_depth (3): 1.5867")
ax.set_xscale("log")
ax.set_xticks(leaf_table["min_samples_leaf"])
ax.set_xticklabels([str(v) for v in leaf_table["min_samples_leaf"]], fontsize=9)
ax.minorticks_off()
ax.set_xlabel("min_samples_leaf (log scale)")
ax.set_ylabel("RMSE")
ax.set_title("A leaf-size limit beats a depth limit here", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**`min_samples_leaf=12` reaches 1.5265 with 13 leaves, beating the best `max_depth` (3) at 1.5867 with
8.**

The margin is small, and the *reason* is the useful part:

> **`max_depth` limits the tree uniformly. `min_samples_leaf` limits it where the data is thin.**

A depth limit forces every branch to stop at the same level, whether it covers a dense region worth
subdividing or a sparse one that is already noise. A leaf-size limit lets the tree go deep where there
are many rows and stop early where there are few - which is a better match to what the tree is actually
trying to do, and gets 13 leaves distributed sensibly rather than 8 distributed evenly.

**Note also the left end.** `min_samples_leaf=1` reproduces the memorising tree exactly: 200 leaves,
training RMSE 0.0000, held-out 2.0709. It is the default.

**In practice, set both.** `min_samples_leaf` for the statistical argument above, and `max_depth` as a
cheap guard on model size and prediction time.

### E10 · How many trees?

In [ ]:
tree_counts = [1, 2, 5, 10, 25, 50, 100, 200, 300]
forest_rows = []
for count in tree_counts:
    forest = RandomForestRegressor(n_estimators=count, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    forest_rows.append({"n_estimators": count,
                        "held-out RMSE": curve_rmse(forest, held_x, held_y)})
forest_table = pd.DataFrame(forest_rows)
print(forest_table.to_string(index=False, float_format=lambda v: "%.4f" % v))
print("\nimprovement from 1 to 25 trees : %.4f"
      % (forest_table["held-out RMSE"].iloc[0] - forest_table["held-out RMSE"].iloc[4]))
print("improvement from 25 to 300     : %.4f"
      % (forest_table["held-out RMSE"].iloc[4] - forest_table["held-out RMSE"].iloc[-1]))

**The curve falls monotonically and flattens: 2.1194 at one tree, 1.7520 at 25, 1.7242 at 300.**

**Almost all of the gain - 0.3675 of 0.3952 - arrives by 25 trees.** The remaining 275 buy 0.0277.

> **`n_estimators` is not a hyperparameter to tune. More is never worse, only slower**, so the rule is:
> set it as high as your time budget allows and never think about it again.

That is genuinely unusual and worth holding onto, because every other dial in this module has an optimum
you can overshoot. Averaging more estimates of the same quantity cannot make the average worse; it can
only make it more precise.

**And an uncomfortable comparison the table invites.** The forest's best is **1.7242**, and the
*single* tree with `min_samples_leaf=12` reached **1.5265**. On this one-dimensional smooth curve a
properly stopped single tree beats a forest of 300, because the forest's trees are grown unrestricted by
default - low bias, high variance - and averaging removes only part of that variance.

**The forest is not the better model here; it is the more robust one.** It got within 13% of the tuned
tree with no tuning at all. On twenty features with interactions, where the depth-3 tree would be
helpless, that ordering reverses - which is the real argument for forests as a default, and not that they
win on every dataset.

### E11 · `max_features`, and what it is buying

In [ ]:
# SYNTHETIC: 800 rows, 20 columns, three real effects and one interaction.
wide_rng = np.random.default_rng(44)
n_wide = 800
wide_X = wide_rng.normal(size=(n_wide, 20))
wide_truth = np.zeros(20)
wide_truth[[0, 3, 7]] = [2.5, -1.8, 1.2]
wide_y = (wide_X @ wide_truth + 1.5 * wide_X[:, 0] * wide_X[:, 3]
          + wide_rng.normal(0, 1.0, n_wide))

wide_train, wide_test, wide_train_y, wide_test_y = train_test_split(
    wide_X, wide_y, test_size=0.3, random_state=0)

feature_rows = []
for how_many in [1, 2, 4, 7, 10, 15, 20]:
    forest = RandomForestRegressor(n_estimators=100, max_features=how_many,
                                   random_state=0).fit(wide_train, wide_train_y)
    per_tree = np.array([tree.predict(wide_test) for tree in forest.estimators_])
    correlations = np.corrcoef(per_tree)
    upper = np.triu_indices_from(correlations, 1)
    feature_rows.append({
        "max_features": how_many,
        "held-out RMSE": float(np.sqrt(((wide_test_y - forest.predict(wide_test)) ** 2).mean())),
        "mean correlation between trees": float(correlations[upper].mean())})
print(pd.DataFrame(feature_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))
print("\nthe noise this data was built with has sd 1.0")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
table = pd.DataFrame(feature_rows)
ax.plot(table["max_features"], table["held-out RMSE"], "o-", color="#D55E00",
        linewidth=2.4, markersize=9, label="held-out RMSE")
ax.set_xlabel("max_features (of 20)")
ax.set_ylabel("held-out RMSE", color="#D55E00")
ax.tick_params(axis="y", labelcolor="#D55E00")

twin = ax.twinx()
twin.plot(table["max_features"], table["mean correlation between trees"], "s--",
          color="#0072B2", linewidth=2.4, markersize=9,
          label="correlation between the trees")
twin.set_ylabel("mean correlation between tree predictions", color="#0072B2")
twin.tick_params(axis="y", labelcolor="#0072B2")

best = table.loc[table["held-out RMSE"].idxmin()]
ax.axvline(best["max_features"], color="#009E73", linewidth=2, alpha=0.6)
ax.text(best["max_features"] + 0.3, table["held-out RMSE"].max() * 0.95,
        "best: %d features" % best["max_features"], color="#009E73", fontsize=10,
        fontweight="bold")
ax.set_title("Fewer features per split means less correlated - and weaker - trees",
             fontsize=11.5)
plt.tight_layout()
plt.show()

**The two curves move together, and the best model is not at either end.**

`max_features=1` gives trees that barely agree with each other - mean correlation **0.0632** - and an
RMSE of **3.1192**. `max_features=20` gives highly correlated trees (0.7871) and 1.3589. The optimum is
**15 features, at RMSE 1.3469 and correlation 0.7471.**

**So "decorrelate the trees" is only half of the mechanism.** Restricting the feature choice does two
things at once, and they pull in opposite directions:

- It makes the trees **less correlated**, so more of their error cancels when averaged. Good.
- It makes each tree **worse**, because it is often forbidden from splitting on the feature that matters.
  Bad.

At `max_features=1` a tree considering only column 11 must split on column 11, however useless it is. The
resulting ensemble is beautifully decorrelated and made of rubbish.

**The optimum trades the two off**, and where it sits depends on how many of the features carry signal.
Here 3 of 20 do, so restricting the choice frequently excludes all three, and the optimum sits high.
**With many redundant informative features, a low `max_features` wins** - which is the situation the
default was designed around.

**A practical note:** scikit-learn's regression default is `max_features=1.0`, that is, all of them - the
right-hand end of this plot, and *not* the `sqrt(p)` often quoted from the classification setting.
Here that default is close to optimal but not quite; it is worth one cross-validated sweep.

### E12 · A counter column, and what a tree does with it

In [ ]:
# SYNTHETIC: 600 shop-days with a genuine upward trend of 0.25 per day.
trend_rng = np.random.default_rng(9)
n_trend = 600
day_number = np.arange(n_trend).astype(float)
trend_shop = pd.DataFrame({
    "footfall": trend_rng.uniform(50, 400, n_trend),
    "promo": trend_rng.integers(0, 2, n_trend).astype(float),
    "weekend": trend_rng.integers(0, 2, n_trend).astype(float),
    "day_number": day_number})
trend_sales = (200 + 0.9 * trend_shop.footfall + 15 * trend_shop.promo
               + 40 * trend_shop.weekend
               + 120 * trend_shop.promo * trend_shop.weekend
               + 0.25 * day_number + trend_rng.normal(0, 25, n_trend))

CUT = 450                                   # train on days 0-449, test on 450-599
rows = []
for label, columns, model in [
        ("forest, with day_number", list(trend_shop.columns),
         RandomForestRegressor(n_estimators=300, random_state=0)),
        ("forest, without it", ["footfall", "promo", "weekend"],
         RandomForestRegressor(n_estimators=300, random_state=0)),
        ("linear, with day_number", list(trend_shop.columns), LinearRegression())]:
    model.fit(trend_shop[columns][:CUT], trend_sales[:CUT])
    inside = float(np.sqrt(((trend_sales[:CUT]
                             - model.predict(trend_shop[columns][:CUT])) ** 2).mean()))
    beyond = float(np.sqrt(((trend_sales[CUT:]
                             - model.predict(trend_shop[columns][CUT:])) ** 2).mean()))
    rows.append({"model": label, "RMSE on days 0-449": inside,
                 "RMSE on days 450-599": beyond, "ratio": beyond / inside})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.3f" % v))
print("\nthe noise this data was built with has sd 25")

**The forest looks four times better in-period and loses out of period.**

| | days 0-449 | days 450-599 |
|---|---|---|
| forest, with `day_number` | **10.670** | 45.868 |
| forest, without it | 17.186 | **87.426** |
| linear, with `day_number` | 38.912 | **41.562** |

**The forest with the counter degrades by a factor of 4.3** - 10.67 to 45.87 - because beyond day 449
every row lands in the leaf built from the last stretch of training days, and the trend simply stops.
The model has learned the trend as a staircase and has run out of steps.

**The linear model degrades by a factor of 1.07** and wins out of period despite being much worse inside
it. It represents the trend as a slope, which continues.

**And the forest without the counter is worst of all** at 87.43, which is the point of including that
row: **the answer is not to remove the trending feature.** The trend is real and worth 37.5 units of
sales over the test period; a model that cannot see it is simply wrong.

**What to actually do**, in the order worth trying:

1. **Detrend, then let the tree work on the residual.** Fit the trend with a linear model, subtract it,
   fit the forest to what is left, add the trend back at prediction time. This gets the tree's
   interactions and the line's extrapolation.
2. **Replace the level with something bounded** - `day of week`, `month`, `days since last promotion` -
   which never leaves its training range.
3. **Retrain often, and monitor.** If the model is refitted weekly the extrapolation gap stays small.
   This is the pragmatic answer and it depends on an operational commitment, not a modelling one.

**And the check that would have caught it: split chronologically, not randomly.** A random split puts
days from the whole period in both halves, so nothing extrapolates and the forest's 10.67 looks like the
truth. That is 04-04's argument arriving with a number attached.

## Interpretation

### E13 · The forest beats the linear model by a lot

**Three reasons, and they need different responses:**

**1. A non-linearity the linear model was not given.** The relationship bends and nobody added the term.
**Test:** plot the linear model's residuals against each feature (05-05). A ∪ or a fan names the column.

**2. An interaction the linear model was not given.** The chapter's shop data: a tree reached 27.94 with
nothing supplied, against 40.65 for main effects. **Test:** the non-parallel-lines plot from 05-07, on the
pairs you have reason to suspect - or just add the products the forest's structure suggests and refit the
linear model.

**3. Robustness to something the linear model is sensitive to** - outliers, a skewed feature, an unscaled
column. Trees are invariant to monotone transforms and split on rank, so a feature with a long tail
costs them nothing. **Test:** refit the linear model on rank-transformed or winsorised features and see
how much of the gap closes.

**The cheapest way to distinguish them all: add the terms and see.** If the linear model with a handful of
squared terms and interactions catches the forest, the answer was 1 or 2 and you now have an
interpretable model. If it does not, the forest is using something more complicated, and that is worth
knowing before you decide which to ship.

**And a fourth possibility to rule out first:** the forest is leaking. If it has access to a column the
linear model does not, or the split was random over grouped rows, the comparison is not about model
class at all.

### E14 · The top feature is `customer_id`

**What has happened: an identifier is being used as a lookup key, and the impurity measure is rewarding
it for it.**

Two mechanisms, both present:

- **It is memorising.** With enough depth, splitting on customer id isolates individual customers and
  predicts their historical mean. That is a perfect fit on training rows and worthless on a new customer.
- **The importance measure is biased towards it regardless.** The chapter measured this: a pure-noise
  continuous column outscored a real binary feature, because a column with many distinct values offers
  many split points. An id has the most possible values of anything in the table.

**What I would do:**

1. **Remove it and refit.** An id is not a feature; it is a row label. If performance collapses, the model
   was memorising and the previous score was fiction.
2. **Check the split.** If the same customer appears in training and test, the memorisation is being
   scored as success - 04-04's grouped split, and the fix is `GroupKFold` on the customer.
3. **Replace it with what it was standing in for.** Customer id was probably a proxy for segment, tenure,
   region or historical volume. Those are features; the id is not.
4. **Recheck the importances afterwards** with permutation importance on held-out rows, which would not
   have been fooled in the same way.

## Debugging

### E15 · Good in backtesting, degrading over six months

**Most likely cause: a feature that drifts outside its training range - and the model is a tree, which
cannot extrapolate.**

The chapter's measurement: past the training range a tree returns a constant, and a forest of 200
returns the same constant. E12's version degraded by a factor of 4.3 over 150 days. **Steady degradation
over months, rather than a sudden break, is the signature** - the further the feature drifts, the worse
the frozen prediction gets.

**The check, in one plot:** for each feature, the distribution in the training data against the
distribution in recent production rows. Anything whose recent values sit at or beyond the training
maximum is the culprit. A counter, a date, a cumulative total, a price, a volume.

**Two other causes worth ruling out**, both of which look similar from the metric alone:

- **The relationship changed**, not just the range - a policy change, a new competitor, a pipeline
  change. Compare the model's error on old and new rows *within* the same feature range to separate this
  from extrapolation.
- **A feature's meaning changed upstream** while its name stayed the same, which 05-08's failure lab
  covered.

### E16 · Training R² 0.98, held-out 0.55

**In this order, cheapest and most likely first:**

1. **Set a stopping rule.** The default is `max_depth=None`, which memorises - training RMSE 0.0000 in the
   chapter. Try `min_samples_leaf` in the range 5 to 20 and sweep it. This alone often closes most of the
   gap and costs one line.
2. **Check the split.** A gap this size is also what a grouped or chronological structure produces under
   a random split (04-04). Reshuffle a few times; if the gap is stable it is the model, if it moves the
   split is the problem.
3. **Check `n_estimators` is not 1.** A "forest" of one tree is a tree.
4. **Draw the learning curve** (05-08). A large gap that is still closing means more rows would help; a
   large gap that has stopped closing means the capacity is wrong.
5. **Look for leakage.** Training R² of 0.98 on real data is high enough to be worth checking
   independently of the gap.

**What I would not do first: tune `max_features` or add trees.** Neither addresses memorisation, and more
trees provably cannot - the chapter measured the improvement from 25 to 300 trees at 0.0277 RMSE.

## Exam and interview reasoning

### E17 · "How does a decision tree decide where to split?"

> "For regression it tries every threshold on every feature and picks the one that minimises the total
> squared error inside the two resulting groups - equivalently, the split that removes the most variance.
> Each leaf then predicts the mean of the training targets in it. It repeats that inside each group until
> a stopping rule fires, and if you do not set one it keeps going until every leaf holds a single row,
> which memorises the training data perfectly.
>
> Because the criterion only ever compares a feature to a threshold, nothing needs scaling and any
> monotone transform of a feature gives the identical tree."

**"And why does a random forest use a random subset of features at each split?"**

> "To make the trees disagree. Averaging removes the part of the error the trees do not share, so the
> benefit depends on how *un*correlated they are - and bagging alone leaves them very similar, because
> bootstrap samples overlap heavily and every tree tends to pick the same strong feature at the root.
> Restricting the feature choice forces different trees down different paths.
>
> It is a trade, though: each tree gets worse because it is sometimes forbidden from splitting on the
> feature that matters. I measured this on twenty columns - `max_features=1` gave a mean correlation of
> 0.06 between trees and an RMSE of 3.12, while 15 features gave a correlation of 0.75 and an RMSE of
> 1.35. The optimum is wherever those two effects balance, which depends on how many features carry
> signal."

**What is being tested:** the first answer should reach "mean in the leaf" and "it memorises by default"
without prompting. The follow-up separates people who have read that forests decorrelate the trees from
people who know *why that helps* and what it costs.

## Transfer to a different situation

### E18 · Energy demand from weather, calendar and installed capacity

**Handle the three groups completely differently, and the third is the reason a forest alone will not do.**

**Weather - give it to the tree unchanged.** Temperature's effect on demand is strongly non-linear
(heating below, cooling above, a flat middle) and interacts with the calendar. This is exactly what trees
are for, and no transform or interaction term is needed. Temperature also stays within its historical
range in any given climate, so extrapolation is not a concern.

**Calendar - encode it as bounded, repeating quantities.** Hour of day, day of week, month, holiday flag.
These never leave their training range by construction, and their effects are interactions with weather -
a hot Sunday afternoon is not a hot Tuesday morning. Trees find that unprompted.

**Installed capacity - this is the problem, and it must not go into the tree as a level.** It rises
monotonically and every future value is outside the training range, so a tree will freeze at whatever it
learned about the most recent period, exactly as E12's `day_number` did. Three options:

1. **Model demand *per unit of capacity*** - make the target a ratio - so the trending quantity leaves the
   model entirely and comes back as a multiplier at prediction time.
2. **Detrend:** fit the capacity effect with a linear model, and give the forest the residual.
3. **Use a hybrid** - a linear term for capacity plus a tree for everything else. That is what gradient
   boosting with a linear base learner does, and 05-11 is the chapter for it.

**Why a forest alone is not enough, in one sentence:** the problem contains one feature that requires
extrapolation and several that require flexible interactions, and no single model class in this chapter
does both.

**And what I would evaluate on: a chronological split.** A random split hides the entire capacity problem,
and would report a model that is about to degrade every month it runs.

## Explain it to someone non-technical

### E19 · A tree, and then a forest

> A decision tree is a flowchart of yes/no questions. "Is it the weekend? If yes, is there a promotion?"
> Each answer narrows things down, and at the end it gives you the average of every past day that
> answered the same way. The computer works out which questions to ask by trying every possible question
> and keeping whichever one separates the outcomes best.
>
> The catch is that a single flowchart is fragile - change a few days of history and you get quite a
> different set of questions. So a random forest builds hundreds of them, each on a slightly different
> sample, and averages the answers. The individual disagreements cancel out and what is left is what they
> all agree on.

*(112 words - over the limit, and the second paragraph is the part that earns the overrun.)* If it must
fit in 90, cut the sentence about how questions are chosen: the fragility-and-averaging idea is what
makes a forest make sense, and the split criterion is detail.

## Optional challenge

### E20 · A depth-2 tree from scratch

In [ ]:
class TinyTree:
    # a regression tree of depth at most 2, built with best_split

    def fit(self, x, y):
        self.root_threshold, _ = best_split(x, y)
        self.branches = {}
        for side in ("left", "right"):
            mask = x < self.root_threshold if side == "left" else x >= self.root_threshold
            sub_x, sub_y = x[mask], y[mask]
            threshold, _ = best_split(sub_x, sub_y) if len(sub_x) > 1 else (None, 0.0)
            if threshold is None:
                self.branches[side] = ("leaf", float(sub_y.mean()))
            else:
                left = sub_y[sub_x < threshold]
                right = sub_y[sub_x >= threshold]
                self.branches[side] = ("split", threshold, float(left.mean()),
                                       float(right.mean()))
        return self

    def predict(self, x):
        out = np.empty(len(x))
        for index, value in enumerate(np.asarray(x).ravel()):
            side = "left" if value < self.root_threshold else "right"
            node = self.branches[side]
            if node[0] == "leaf":
                out[index] = node[1]
            else:
                out[index] = node[2] if value < node[1] else node[3]
        return out


agreement = []
for seed in range(5):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-3, 3, 120)
    y = np.sin(x) * 4 + rng.normal(0, 0.7, 120)
    mine = TinyTree().fit(x, y).predict(x)
    theirs = DecisionTreeRegressor(max_depth=2).fit(x.reshape(-1, 1), y).predict(
        x.reshape(-1, 1))
    agreement.append({"seed": seed, "largest disagreement": np.abs(mine - theirs).max()})
print(pd.DataFrame(agreement).to_string(index=False, float_format=lambda v: "%.2e" % v))

**Agreement to machine precision on all five datasets - the largest disagreement anywhere is
1.8e-15.**

Thirty lines reproduce `DecisionTreeRegressor(max_depth=2)` exactly, which is a fair measure of how
little there is to the algorithm. What a production implementation adds is **speed** (sorting once and
updating the sums incrementally, rather than recomputing every group mean from scratch), **more stopping
rules**, **multiple features**, **missing-value handling**, and **pruning** - engineering rather than
ideas.

**The recursion is the only conceptually interesting part**, and it is three lines: split, then solve the
same problem inside each half. Making `TinyTree` handle arbitrary depth is a matter of calling itself
instead of stopping.

### E21 · Where the variance reduction comes from

The standard result for the variance of an average of `k` estimators, each with variance `sigma²` and
pairwise correlation `rho`, is:

$$\text{Var}\left(\text{average}\right) = \rho\,\sigma^2 + \frac{1 - \rho}{k}\,\sigma^2$$

**Read the two terms.** The second vanishes as `k` grows - that is the part more trees buy you. **The
first does not.** However many trees you average, a floor of `rho * sigma²` remains, set entirely by how
correlated the trees are.

> **That is why `max_features` exists.** Adding trees attacks the second term and saturates - the
> chapter measured 0.0278 RMSE from 25 trees to 300. Decorrelating them attacks the *floor*, which is
> the only term left once `k` is large.

And it explains the chapter's factor of 2.14 rather than 100. Bootstrap samples of the same data overlap
heavily, so `rho` is substantial, and with one feature there was nothing for `max_features` to subset.
**Bagging without decorrelation runs into the floor almost immediately** - which is precisely the
observation that turned bagging into the random forest.